# Downtown Edmonton Business Classifier

This notebook classifies companies as **downtown Edmonton** or not based on their Canadian postal codes.

### How it works
Canadian postal codes follow the format `A1A 1A1`. The first three characters form the **Forward Sortation Area (FSA)**, which maps to a specific geographic zone. The FSAs for downtown Edmonton are:

| FSA | Neighbourhood |
|-----|---------------|
| T5H | Downtown core (south) |
| T5J | Downtown core (financial/central district) |
| T5K | Oliver neighbourhood (immediately west of downtown) |

In [1]:
# Install dependencies if needed
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'openpyxl', 'pandas', '-q'])

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', 'openpyxl', 'pandas', '-q'], returncode=0)

In [2]:
import pandas as pd

# ── Configuration ────────────────────────────────────────────────────────────
DATA_FILE = 'Office_zoom_contact_list.xlsx'

# FSAs (first 3 characters of postal code) that cover downtown Edmonton.
# Adjust this set if you want to widen or narrow the definition of "downtown".
DOWNTOWN_EDMONTON_FSAS = {'T5H', 'T5J', 'T5K'}
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_excel(DATA_FILE)
print(f'Loaded {len(df)} rows')
df.head()

Loaded 101 rows


,Company Name,Company Zip Code
0,Arc Studio,T5N 3N7
1,RCL Canada,T6E 0E4
2,Headwater Engineering,T6B 2R2
3,Continental Chain & Rigging,T4S 2L4
4,Green Analytics,T6E 5K7


In [3]:
def extract_fsa(postal_code) -> str | None:
    """Return the 3-char FSA from a Canadian postal code, or None if invalid."""
    if pd.isna(postal_code):
        return None
    cleaned = str(postal_code).strip().upper().replace('-', ' ')
    # Accept 'A1A 1A1' or 'A1A1A1'
    fsa = cleaned[:3]
    # Basic validation: letter-digit-letter
    if len(fsa) == 3 and fsa[0].isalpha() and fsa[1].isdigit() and fsa[2].isalpha():
        return fsa
    return None


def is_downtown_edmonton(postal_code) -> bool | None:
    """Return True if the postal code falls within downtown Edmonton.
    Returns None when the postal code is missing or unparseable.
    """
    fsa = extract_fsa(postal_code)
    if fsa is None:
        return None
    return fsa in DOWNTOWN_EDMONTON_FSAS


# Derive columns
postal_col = df.columns[1]          # 'Company Zip Code'
company_col = df.columns[0]         # 'Company Name'

df['FSA'] = df[postal_col].apply(extract_fsa)
df['Is Downtown Edmonton'] = df[postal_col].apply(is_downtown_edmonton)

df[[company_col, postal_col, 'FSA', 'Is Downtown Edmonton']].head(10)

,Company Name,Company Zip Code,FSA,Is Downtown Edmonton
0,Arc Studio,T5N 3N7,T5N,False
1,RCL Canada,T6E 0E4,T6E,False
2,Headwater Engineering,T6B 2R2,T6B,False
3,Continental Chain & Rigging,T4S 2L4,T4S,False
4,Green Analytics,T6E 5K7,T6E,False
5,Regent Supply,T6E 5Y8,T6E,False
6,Veritas Solutions,T6E 1T4,T6E,False
7,McArt Consulting,NaN,NaN,None
8,Link Industrial,T6N 1B2,T6N,False
9,Graphic Intuitions,R0G 1K0,R0G,False


In [4]:
# ── Summary ───────────────────────────────────────────────────────────────────
total        = len(df)
downtown     = df['Is Downtown Edmonton'].sum()
not_downtown = (df['Is Downtown Edmonton'] == False).sum()
no_data      = df['Is Downtown Edmonton'].isna().sum()

print(f'Total companies   : {total}')
print(f'Downtown Edmonton : {int(downtown)}')
print(f'Not downtown      : {int(not_downtown)}')
print(f'No postal code    : {int(no_data)}')

Total companies   : 101
Downtown Edmonton : 9
Not downtown      : 74
No postal code    : 18


In [5]:
# ── Downtown companies ────────────────────────────────────────────────────────
print('Companies in downtown Edmonton:')
downtown_df = df[df['Is Downtown Edmonton'] == True][[company_col, postal_col, 'FSA']]
downtown_df = downtown_df.drop_duplicates(subset=company_col).reset_index(drop=True)
downtown_df

Companies in downtown Edmonton:


,Company Name,Company Zip Code,FSA
0,Invistec Consulting,T5J 3N9,T5J
1,Wyvern,T5K 0K6,T5K
2,Intelligence House,T5J 0X6,T5J
3,Hahn & Houle,T5J 3H1,T5J
4,Gameplan HR,T5J 1W8,T5J
5,Chatwin,T5J 3G2,T5J
6,Top Draw,T5K 1K9,T5K


In [6]:
# ── Full results table ────────────────────────────────────────────────────────
result = df[[company_col, postal_col, 'FSA', 'Is Downtown Edmonton']].copy()
result['Is Downtown Edmonton'] = result['Is Downtown Edmonton'].map(
    {True: 'Yes', False: 'No', None: 'Unknown'}
).fillna('Unknown')
result

,Company Name,Company Zip Code,FSA,Is Downtown Edmonton
0,Arc Studio,T5N 3N7,T5N,No
1,RCL Canada,T6E 0E4,T6E,No
2,Headwater Engineering,T6B 2R2,T6B,No
3,Continental Chain & Rigging,T4S 2L4,T4S,No
4,Green Analytics,T6E 5K7,T6E,No
...,...,...,...,...
96,Accelerate Chartered,T6E 6S4,T6E,No
97,Boreal Land,T8A 4H3,T8A,No
98,Preferred Client Services Group,T6X 0J6,T6X,No
99,Nor-Alta Environmental Services,T5P 4W2,T5P,No


In [7]:
# ── Export results ────────────────────────────────────────────────────────────
out_file = 'downtown_edmonton_results.xlsx'
result.to_excel(out_file, index=False)
print(f'Results saved to {out_file}')

Results saved to downtown_edmonton_results.xlsx
